<a href="https://colab.research.google.com/github/Pere91/SAC_Spark/blob/main/SAC_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.7/spark-3.5.7-bin-hadoop3.tgz
!tar xf spark-3.5.7-bin-hadoop3.tgz

E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-17/openjdk-17-jre-headless_17.0.16%2b8%7eus1-0ubuntu1%7e22.04.1_amd64.deb  404  Not Found [IP: 185.125.190.81 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-17/openjdk-17-jdk-headless_17.0.16%2b8%7eus1-0ubuntu1%7e22.04.1_amd64.deb  404  Not Found [IP: 185.125.190.81 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.7-bin-hadoop3"
os.environ["PATH"] += os.pathsep + os.path.join(os.environ["SPARK_HOME"], "bin")

In [5]:
from pyspark import SparkConf, SparkContext
import networkx as nx

In [6]:
conf = SparkConf().setMaster("local").setAppName("SAC_Spark")
sc = SparkContext.getOrCreate(conf=conf)

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
G = nx.DiGraph()
G.add_weighted_edges_from([
    ("A", "C", 3.0), ("A", "F", 2.0),
    ("C", "D", 4.0), ("C", "F", 2.0), ("C", "E", 1.0),
    ("F", "E", 3.0), ("F", "G", 5.0), ("F", "B", 6.0),
    ("E", "B", 2.0),
    ("D", "B", 1.0),
    ("A", "C", 3.0),
    ("G", "B", 2.0)
])

In [ ]:
INIT_NODE = 'A'
pyspark_graph = []
graph_dict = {}

for node in G.nodes():
  neighbors = [(nbr, G.edges[node, nbr]["weight"]) for nbr in G.successors(node)]

  if node == INIT_NODE:
    weight = 0
  else:
    weight = float("inf")

  pyspark_graph.append((node, (neighbors, weight, False, [])))
  graph_dict[node] = (neighbors, weight, False, [])

In [ ]:
for city in pyspark_graph:
  print(city)

In [ ]:
vertices = sc.parallelize(pyspark_graph)
start_node = vertices.filter(lambda x: x[0] == INIT_NODE).collect()